# Glowrithm: training and testing on Google Colab
Runtime > Change runtime type > **T4 GPU**, then run the cells in order. Upload `glowrithm.zip` (this repository) to `MyDrive/glowrithm/` first.

The notebook downloads the Kaggle dataset, audits it, trains the ensemble and the two single-backbone baselines, writes the CD-5 evaluation files and finally serves the **web test build** of the app (section 5) with the trained model, so the whole flow can be tried in a browser.

In [ ]:
!nvidia-smi -L
import tensorflow as tf, keras
print(tf.__version__, keras.__version__, tf.config.list_physical_devices('GPU'))

## 1. Project files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/glowrithm"
!unzip -q -o "$DRIVE_DIR/glowrithm.zip" -d /content
%cd /content/glowrithm/ml
!pip install -q -e . scikit-learn matplotlib pandas pyyaml

## 2. Dataset via the Kaggle API
Kaggle > Settings > API > *Create New Token* downloads `kaggle.json`; upload it when asked. The slug below is only an example: check the dataset page (licence, classes) or replace it with your own dataset. Folder names must contain `dry`, `normal` and `oily`.

In [ ]:
import os
from google.colab import files
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
KAGGLE_DATASET = "shakyadissanayake/oily-dry-and-normal-skin-types-dataset"  # dataset of the initial test
!kaggle datasets download -d $KAGGLE_DATASET -p data/raw --unzip
!find data/raw -maxdepth 3 -type d | head -20

### Dataset audit (before preparing it)
Counts per split and class, image sizes, exact and near-duplicate photos (also mirrored copies and Roboflow-style
augmented copies), photos that appear in more than one split, face-detection rate and quality-gate measurements.
Copy the numbers into the dataset description of CD-4/CD-5.

In [ ]:
!python scripts/audit_dataset.py --raw data/raw --out reports/dataset_audit
from IPython.display import Image, Markdown, display
display(Markdown(open('reports/dataset_audit/audit.md').read()))
display(Image('reports/dataset_audit/sample_grid.png', width=900))

In [ ]:
!python scripts/prepare_dataset.py --config config.yaml
!cat data/processed/summary.json
!python scripts/quality_report.py --config config.yaml --split train   # calibrates the photo-quality gate

## 3. Pipeline check, then training (ensemble + two baselines for the ablation test)

In [ ]:
!python scripts/train.py --config config.yaml --smoke-test --out artifacts/smoke

In [ ]:
!python scripts/train.py --config config.yaml --arch ensemble --verbose 1

In [ ]:
!python scripts/train.py --config config.yaml --arch resnet --verbose 1
!python scripts/train.py --config config.yaml --arch effnet --verbose 1

*No GPU quota?* `!python scripts/train_cached.py --config config.yaml --views 4` trains the heads of all three
models on frozen ImageNet features in about 15-30 minutes on a CPU (no fine-tuning), with the same output files.

## 4. Evaluation for CD-5

In [ ]:
M = 'artifacts/ensemble/model.keras'
!python scripts/evaluate.py --model $M --split test
!python scripts/evaluate.py --model artifacts/resnet/model.keras --split test
!python scripts/evaluate.py --model artifacts/effnet/model.keras --split test
# Is the ensemble significantly better than a single backbone? (McNemar test, CD-5 ablation)
!python scripts/compare_models.py artifacts/ensemble/reports/test/predictions.csv artifacts/resnet/reports/test/predictions.csv --names ensemble resnet
!python scripts/compare_models.py artifacts/ensemble/reports/test/predictions.csv artifacts/effnet/reports/test/predictions.csv --names ensemble effnet
!python scripts/robustness.py --model $M
!python scripts/gradcam_report.py --model $M --per-class 3
!python scripts/xai_deletion.py --model $M --max-images 100
!python scripts/export_tflite.py --model $M --quantize float16
!python scripts/benchmark.py --model $M --tflite artifacts/ensemble/model_float16.tflite

In [ ]:
from IPython.display import Image, display
for f in ['training_curves.png', 'reports/test/confusion_matrix_normalized.png', 'reports/robustness/robustness.png', 'reports/gradcam/gradcam_oily.png']:
    display(Image(f'artifacts/ensemble/{f}', width=700))

## 5. Test the web app with the trained model
Starts the API with the ensemble model and serves the web test build at `/app/`. The first link is a private
Colab proxy (only your Google account can open it). Colab links are HTTPS, so the live camera works.

In [ ]:
import os, subprocess, time, urllib.request
!pip install -q -r ../backend/requirements.txt
!mkdir -p ../backend/models && cp artifacts/ensemble/model.keras artifacts/ensemble/model_meta.json ../backend/models/
server = subprocess.Popen(["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
                          cwd="/content/glowrithm/backend", env={**os.environ, "PYTHONPATH": "/content/glowrithm/ml"},
                          stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8000/api/v1/health", timeout=2).read().decode())
        break
    except Exception:
        time.sleep(2)
from google.colab.output import eval_js
print("Open the web app:", eval_js("google.colab.kernel.proxyPort(8000)") + "app/")

**Optional, for testing on a phone:** a temporary public HTTPS link through a Cloudflare quick tunnel. Anyone
with the link can use the server while it runs, and photos pass through Cloudflare, so use it only with the
team's own test photos and stop it afterwards (`tunnel.terminate()`).

In [ ]:
import re
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("Phone link:", match.group(0) + "/app/")
        break

## 6. Save everything to Google Drive

In [ ]:
!rm -rf artifacts/smoke
!mkdir -p "$DRIVE_DIR/artifacts" && cp -r artifacts/* "$DRIVE_DIR/artifacts/"
!cp data/processed/summary.json "$DRIVE_DIR/artifacts/" && cp -r reports "$DRIVE_DIR/"
print("Copied to", DRIVE_DIR)